# 예스24 오픈 API — 베스트셀러 데이터 분석

국내도서 베스트셀러 200건을 수집해 pandas 로 분석하는 예제입니다.

1. **수집** — `GET /v1/category/bestseller` (100건 × 2페이지, 요청 한도 준수)
2. **분석** — 판매가·할인율 분포, 출판사 TOP 10, 출간 연도 분포

**준비**
- `pip install -r requirements.txt`
- 환경변수 `YES24_API_KEY` 설정 (발급: [예스24 개발자센터](https://developers.yes24.com))

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests

API_BASE = os.environ.get("YES24_API_BASE_URL", "https://apis.yes24.com")
API_KEY = os.environ.get("YES24_API_KEY")
assert API_KEY, "환경변수 YES24_API_KEY 를 설정하세요. 발급: https://developers.yes24.com"

# matplotlib 한글 깨짐 방지 — OS별 기본 한글 폰트
if sys.platform.startswith("win"):
    plt.rcParams["font.family"] = "Malgun Gothic"
elif sys.platform == "darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"  # 리눅스는 나눔고딕 설치 필요
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 수집

요청 한도(5회/초)를 지키기 위해 페이지 사이에 `time.sleep(0.25)` 을 넣고,
429 응답이 오면 `Retry-After` 헤더가 안내하는 시간만큼 기다렸다가 재시도합니다.

In [ ]:
CATEGORY_ID = "001"  # 국내도서
PAGE_SIZE = 100
PAGES = 2

session = requests.Session()
session.headers["X-Api-Key"] = API_KEY

rows = []
page = 1
while page <= PAGES:
    resp = session.get(
        f"{API_BASE}/v1/category/bestseller",
        params={"categoryId": CATEGORY_ID, "page": page, "pageSize": PAGE_SIZE},
        timeout=10,
    )
    if resp.status_code == 429:
        wait = int(resp.headers.get("Retry-After") or 1)
        print(f"요청 한도 초과(429) — {wait}초 대기 후 재시도")
        time.sleep(wait)
        continue

    payload = resp.json()
    assert payload.get("success"), f"API 오류 [{payload.get('errorCode')}] {payload.get('message')}"

    rows.extend(payload["data"]["items"])
    print(f"{page}페이지 수집 완료 (누적 {len(rows)}건)")
    page += 1
    time.sleep(0.25)

In [ ]:
df = pd.json_normalize(rows)
df["shopPrice"] = pd.to_numeric(df["shopPrice"], errors="coerce")
df["salePrice"] = pd.to_numeric(df["salePrice"], errors="coerce")
df["publishYear"] = pd.to_datetime(df["publishDate"], errors="coerce").dt.year
priced = df["shopPrice"] > 0
df.loc[priced, "discountRate"] = (
    (df.loc[priced, "shopPrice"] - df.loc[priced, "salePrice"]) / df.loc[priced, "shopPrice"] * 100
).round(1)

df[["sortOrder", "title", "author", "publisher", "salePrice", "discountRate", "publishDate", "upDown"]].head(10)

## 2. 판매가·할인율 분포

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.hist(df["salePrice"].dropna() / 1000, bins=15, color="#4c78a8", edgecolor="white")
ax1.set_title("판매가 분포")
ax1.set_xlabel("판매가 (천 원)")
ax1.set_ylabel("도서 수")
ax2.hist(df["discountRate"].dropna(), bins=12, color="#f58518", edgecolor="white")
ax2.set_title("할인율 분포")
ax2.set_xlabel("할인율 (%)")
ax2.set_ylabel("도서 수")
fig.suptitle(f"베스트셀러 가격 분석 (표본 {len(df)}건)")
fig.tight_layout()
plt.show()

print(f"평균 판매가 {df['salePrice'].mean():,.0f}원 / 중앙값 {df['salePrice'].median():,.0f}원 / 평균 할인율 {df['discountRate'].mean():.1f}%")

## 3. 출판사 TOP 10

In [ ]:
top = df["publisher"].value_counts().head(10).sort_values()
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(top.index, top.values, color="#4c78a8")
ax.set_title("베스트셀러 최다 진입 출판사 TOP 10")
ax.set_xlabel("도서 수")
fig.tight_layout()
plt.show()

## 4. 출간 연도 분포와 순위 등락

In [ ]:
counts = df["publishYear"].dropna().astype(int).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(counts.index.astype(str), counts.values, color="#54a24b")
ax.set_title("베스트셀러 출간 연도 분포")
ax.set_xlabel("출간 연도")
ax.set_ylabel("도서 수")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
plt.show()

up = int((df["upDown"] > 0).sum())
down = int((df["upDown"] < 0).sum())
same = int((df["upDown"] == 0).sum())
print(f"순위 등락 — 상승 {up} / 하락 {down} / 유지 {same}")

## 심화 과제

- `GET /v1/category/bestsellerDaily` 의 `date` 파라미터로 여러 날짜를 수집해 특정 도서의 **순위 변동 추이**를 그려 보세요. (요청 한도를 고려해 날짜 수를 조절하세요)
- `sex`(성별)·`age`(연령) 필터를 바꿔 **독자층별 베스트셀러 차이**를 비교해 보세요.
- `detail=Y` 로 조회하면 페이지 수·평점 등 확장 필드로 더 깊은 분석이 가능합니다.